In [7]:
from transformers import Wav2Vec2Processor, Wav2Vec2Model
from datasets import load_dataset
import torch
 
# load model and processor
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-large-960h")
model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-large-960h")
    
# load dummy dataset and read soundfiles
ds = load_dataset("patrickvonplaten/librispeech_asr_dummy", "clean", split="validation")

# tokenize
input_values = processor(ds[0]["audio"]["array"], return_tensors="pt", padding="longest", sampling_rate=16000).input_values  # Batch size 1

# # retrieve logits
# logits = model(input_values).logits

# # take argmax and decode
# predicted_ids = torch.argmax(logits, dim=-1)
# transcription = processor.batch_decode(predicted_ids)
outputs = model(input_values)
features = outputs.last_hidden_state  # shape: (B, T, H)



Some weights of the model checkpoint at facebook/wav2vec2-large-960h were not used when initializing Wav2Vec2Model: ['lm_head.weight', 'lm_head.bias']
- This IS expected if you are initializing Wav2Vec2Model from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing Wav2Vec2Model from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Found cached dataset librispeech_asr_dummy (/home/you/.cache/huggingface/datasets/patrickvonplaten___librispeech_asr_dummy/clean/2.1.0/f2c70a4d03ab4410954901bde48c54b85ca1b7f9bf7d616e7e2a72b5ee6ddbfc)


In [8]:
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
from datasets import load_dataset
import torch
import torch.nn as nn
import torch.nn.functional as F

class Wav2Vec2AVDHead(nn.Module):
    def __init__(self, base_model_name="facebook/wav2vec2-large-960h"):
        super().__init__()
        self.feature_extractor = Wav2Vec2Model.from_pretrained(base_model_name)
        hidden_size = self.feature_extractor.config.hidden_size  # typically 1024
        self.projection = nn.Linear(hidden_size, 3)  # A, V, D

    def forward(self, input_values):
        with torch.no_grad():  # 特征部分不训练
            features = self.feature_extractor(input_values).last_hidden_state  # (B, T, H)
        pooled = features.mean(dim=1)  # average pooling over time → (B, H)
        avd = torch.sigmoid(self.projection(pooled))  # scale to [0, 1]
        return avd


# 加载 processor
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-large-960h")

# 使用官方提供的 demo 数据
ds = load_dataset("patrickvonplaten/librispeech_asr_dummy", "clean", split="validation")
waveform = ds[0]["audio"]["array"]
sampling_rate = ds[0]["audio"]["sampling_rate"]


# 准备输入
inputs = processor(waveform, sampling_rate=sampling_rate, return_tensors="pt").input_values

# 初始化模型
model = Wav2Vec2AVDHead()

# 推理
model.eval()
with torch.no_grad():
    avd_vector = model(inputs).squeeze().tolist()
    print("Predicted AVD:", avd_vector)



Found cached dataset librispeech_asr_dummy (/home/you/.cache/huggingface/datasets/patrickvonplaten___librispeech_asr_dummy/clean/2.1.0/f2c70a4d03ab4410954901bde48c54b85ca1b7f9bf7d616e7e2a72b5ee6ddbfc)
Some weights of the model checkpoint at facebook/wav2vec2-large-960h were not used when initializing Wav2Vec2Model: ['lm_head.weight', 'lm_head.bias']
- This IS expected if you are initializing Wav2Vec2Model from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing Wav2Vec2Model from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Predicted AVD: [0.459077388048172, 0.48602429032325745, 0.48193421959877014]
